# 01 – Veri, Bölünme ve Keşifsel Veri Analizi (EDA)
**Yapay Sinir Ağları – Yüksek Lisans Ara Sınav Projesi**
Hazırlayan(lar), ders bilgisi ve seed kaynağı: `configs/project.json`

Bu defter şunları yapar:
1. Ortam/sistem bilgisi ve seed'lerin temel seed'den türetilmesi (`configs/seeds.json`)
2. `bank-additional.csv` indirme, değişken/etiket kontrolleri, `duration` çıkarımı
3. Kopya kayıt incelemesi
4. Kopya grubuna duyarlı, stratified %80 Development / %20 Test ayrımı (`results/dataset_split.csv`)
5. **Yalnızca Development** üzerinde EDA (tablolar + Şekil 2a–2g) ve deney hipotezleri

> Test verisi bu defterde yalnızca diske ayrı bir dosya olarak yazılır; hiçbir istatistik, grafik veya karar için kullanılmaz.

In [ ]:
# === 0. Ortam kurulumu (üç defterde aynı) ===
import os, sys, subprocess, importlib

# Kendi GitHub reponuzun adresi (boş bırakılırsa Drive'daki YSA_Midterm_Project klasörü kullanılır)
GITHUB_REPO_URL = ""   # ör. "https://github.com/<kullanici-adi>/ysa-mlp-bank-marketing-suleyman-balsak.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if GITHUB_REPO_URL:
        repo = GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
        PROJECT_DIR = f"/content/drive/MyDrive/{repo}"
        if not os.path.exists(os.path.join(PROJECT_DIR, "src")):
            subprocess.run(["git", "clone", GITHUB_REPO_URL, PROJECT_DIR], check=True)
        else:   # yalnızca kod güncellemelerini çeker; üretilen sonuçlara dokunmaz
            r = subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], capture_output=True, text=True)
            print("git pull:", (r.stdout or r.stderr).strip()[:200])
    else:
        PROJECT_DIR = "/content/drive/MyDrive/YSA_Midterm_Project"
else:  # yerel Jupyter: defter proje kök dizininde çalıştırılır
    PROJECT_DIR = os.getcwd()

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for d in ["configs", "results", "results/cache", "figures", "data", "models"]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists("src/ysa_utils.py"):
    raise FileNotFoundError(
        f"{PROJECT_DIR}/src/ysa_utils.py bulunamadı!\n"
        "Çözüm 1: GITHUB_REPO_URL'ye kendi repo adresinizi yazın (repo src/ klasörünü içermeli).\n"
        f"Çözüm 2: Drive'da {PROJECT_DIR}/src/ klasörü açıp ysa_utils.py dosyasını oraya yükleyin.")

import src.ysa_utils as U
importlib.reload(U)                      # modül değişirse yeniden yükle
from src.ysa_utils import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Çalışma dizini:", os.getcwd())

## 1. Sistem bilgisi ve seed'ler

In [ ]:
import platform, json, torch, sklearn, matplotlib

assert os.path.exists("configs/project.json"), "configs/project.json bulunamadı (yazarlar ve seed kaynağı burada)."
PROJECT = load_json("configs/project.json")
SEEDS = derive_seeds(int(PROJECT["student_id_last5"]))     # temel seed = son 5 hane + 2026
print(f"Proje türü: {PROJECT['project_type']} | Hazırlayan(lar): "
      + ", ".join(f"{a['name']} ({a['student_id']})" for a in PROJECT["authors"]))
print(f"Seed kaynağı: {PROJECT['seed_owner']} → son 5 hane {PROJECT['student_id_last5']} → temel seed {SEEDS['base_seed']}")
if PROJECT.get("seed_note"):
    print("Seed notu:", PROJECT["seed_note"])
save_json(SEEDS, "configs/seeds.json")

system_info = {
    "os": platform.platform(),
    "processor": platform.processor() or platform.machine(),
    "cpu_count": os.cpu_count(),
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__,
    "seaborn_version": sns.__version__,
    "training_device": TRAINING_SETTINGS["device"],
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
    "environment": "Google Colab" if IN_COLAB else "Yerel Jupyter",
}
save_json(system_info, "results/system_info.json")
print(json.dumps(SEEDS, indent=2))
print(json.dumps(system_info, indent=2, ensure_ascii=False))

## 2. Veri setinin indirilmesi ve temel kontroller

In [ ]:
import urllib.request, zipfile

CSV_PATH = "data/bank-additional/bank-additional.csv"
URLS = [
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip",
    "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip",   # yedek (iç içe zip)
]

def _extract_bank_additional(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        if any(n.endswith("bank-additional/bank-additional.csv") for n in names):
            z.extractall("data")
            return True
        inner = [n for n in names if n.endswith("bank-additional.zip")]
        if inner:
            z.extract(inner[0], "data")
            return _extract_bank_additional(os.path.join("data", inner[0]))
    return False

if not os.path.exists(CSV_PATH):
    for url in URLS:
        try:
            zp = os.path.join("data", os.path.basename(url).replace("+", "_"))
            urllib.request.urlretrieve(url, zp)
            if _extract_bank_additional(zp):
                print("İndirildi:", url)
                break
        except Exception as e:
            print("İndirme başarısız:", url, "→", e)
assert os.path.exists(CSV_PATH), "bank-additional.csv bulunamadı; dosyayı data/bank-additional/ altına elle koyun."

raw = pd.read_csv(CSV_PATH, sep=";")
print("Ham veri boyutu:", raw.shape)
assert raw.shape == (4119, 21), "Beklenen boyut (4119, 21) değil – yanlış dosya (ör. -full sürümü) olabilir!"

EXPECTED = ["age", "job", "marital", "education", "default", "housing", "loan", "contact", "month",
            "day_of_week", "duration", "campaign", "pdays", "previous", "poutcome", "emp.var.rate",
            "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed", "y"]
assert list(raw.columns) == EXPECTED, "Değişken adları/sırası beklenenden farklı!"
assert set(raw["y"].unique()) == {"yes", "no"}, "Beklenmeyen hedef etiketi!"
print("Gerçek NaN sayısı:", int(raw.isna().sum().sum()))
print("\nHedef dağılımı (ham, tüm veri – yalnızca okuma kontrolü):")
print(raw["y"].value_counts())

df = raw.copy()
df["y"] = (df["y"] == "yes").astype(int)          # yes → 1 (pozitif), no → 0
df = df.drop(columns=DROP_COLS)                    # Bölüm 2.2 uygulama kuralı
df.insert(0, ID_COL, raw.index)                    # orijinal satır indeksi korunur
print("\n'duration' çıkarıldıktan sonra:", df.shape, "→ 19 girdi + y + original_index")
print(df.dtypes.value_counts())

## 3. Kopya kayıtların incelenmesi (Bölüm 3.4)
Veri setinde müşteri kimliği yoktur. Bu nedenle tamamen aynı satırların aynı müşterinin tekrar kaydı mı yoksa tesadüfen aynı profile sahip farklı müşteriler mi olduğu ayırt edilemez.

**Uygulanan kural:**
1. Kopyalar **silinmez** (dağılımı keyfi biçimde değiştirmemek için).
2. `duration` hariç tüm girdileri aynı olan satırlara ortak bir `dup_group` kimliği verilir.
3. **Bütün bölünmeler bu gruba duyarlıdır.** Dev/test ayrımı, outer/inner CV katları ve early-stop alt kümesi aynı kopyayı iki tarafa bölmez. Böylece bir kopyanın eğitimde görülüp değerlendirmede "yeni örnek" gibi ölçülmesi engellenir.

Kopya olmasaydı bu yöntemin sonucu sıradan stratified bölünmeyle birebir aynı olurdu.

In [ ]:
FEATS = [c for c in df.columns if c not in [ID_COL, TARGET]]

n_dup_raw = int(raw.duplicated().sum())                          # duration dahil ham veri
n_dup_full = int(df.duplicated(subset=FEATS + [TARGET]).sum())   # duration hariç, y dahil
n_dup_X = int(df.duplicated(subset=FEATS).sum())                 # yalnızca özellikler

df[GROUP_COL] = duplicate_groups(df, FEATS).values
grp_size = df.groupby(GROUP_COL)[ID_COL].transform("size")
conflict = df.groupby(GROUP_COL)[TARGET].transform("nunique") > 1

dup_summary = pd.DataFrame([
    {"Kontrol": "Ham veride (duration dahil) tam kopya satır", "Sayı": n_dup_raw},
    {"Kontrol": "duration çıkarıldıktan sonra kopya (özellikler + y)", "Sayı": n_dup_full},
    {"Kontrol": "Yalnızca özellikler bazında kopya", "Sayı": n_dup_X},
    {"Kontrol": "Kopya grubuna ait toplam satır", "Sayı": int((grp_size > 1).sum())},
    {"Kontrol": "Etiketi çelişen (aynı X, farklı y) satır", "Sayı": int(conflict.sum())},
])
print(dup_summary.to_string(index=False))

## 4. Stratified (kopya grubuna duyarlı) %80 Development / %20 Test ayrımı

In [ ]:
dev_pos, test_pos = group_aware_holdout(df[TARGET].values, df[GROUP_COL].values, 0.20, SEEDS["split_seed"])
dev_df, test_df = df.iloc[dev_pos].copy(), df.iloc[test_pos].copy()

# Kontrol: hiçbir kopya grubu iki tarafa bölünmemeli
dup_groups = df.loc[grp_size > 1, GROUP_COL].unique()
n_span = len(np.intersect1d(dev_df[GROUP_COL], test_df[GROUP_COL]))
assert n_span == 0, "Kopya grubu dev ve test arasında bölünmüş!"
dup_summary.loc[len(dup_summary)] = {"Kontrol": "Hem dev hem test'e düşen kopya grubu (grup-duyarlı bölünme sonrası)", "Sayı": n_span}
dup_summary.to_csv("results/duplicate_check.csv", index=False)
print(f"Toplam kopya grubu: {len(dup_groups)} | Dev–Test arasında bölünen grup: {n_span} | "
      f"dev oranı = {len(dev_df) / len(df):.4f}")

# dataset_split.csv: orijinal satır indeksi ve atandığı küme (Bölüm 3.3)
split_record = pd.DataFrame({
    ID_COL: df[ID_COL],
    "split": np.where(df[ID_COL].isin(dev_df[ID_COL]), "development", "test"),
    GROUP_COL: df[GROUP_COL],
}).sort_values(ID_COL)
split_record.to_csv("results/dataset_split.csv", index=False)

split_summary = pd.DataFrame([
    {"Küme": name, "n": len(s), "Pozitif (y=1)": int(s[TARGET].sum()),
     "Negatif (y=0)": int((s[TARGET] == 0).sum()), "Pozitif oranı %": round(100 * s[TARGET].mean(), 2),
     "Seed": SEEDS["split_seed"]}
    for name, s in [("Tüm veri", df), ("Development (%80)", dev_df), ("Independent Test (%20)", test_df)]
])
split_summary.to_csv("results/dataset_split_summary.csv", index=False)
print(split_summary.to_string(index=False))

dev_df.to_csv("data/dev_data.csv", index=False)      # dup_group sütunu 02/03'te bölünmeler için kullanılır
test_df.to_csv("data/test_data.csv", index=False)
del test_df          # bu defterin geri kalanında test verisi bellekte tutulmaz

---
# EDA – yalnızca Development verisi (n ≈ 3.295)
> Buradaki hedef–özellik karşılaştırmaları **betimseldir**. EDA'ya bakılarak özellik elenmez veya imputation stratejisi seçilmez (Bölüm 4 notu). Alternatifler (unknown stratejisi, scaler, clipping) nested CV içinde değerlendirilir.

In [ ]:
X_eda = dev_df.drop(columns=[ID_COL, TARGET, GROUP_COL])
y_eda = dev_df[TARGET]
num_cols_raw = X_eda.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_raw = X_eda.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Development n = {len(X_eda)} | Sayısal ({len(num_cols_raw)}): {num_cols_raw}")
print(f"Kategorik ({len(cat_cols_raw)}): {cat_cols_raw}")

# Tablo 1a – özellik genel bakış
overview = []
for c in X_eda.columns:
    s = X_eda[c]
    overview.append({
        "Özellik": c, "Tür": "sayısal" if c in num_cols_raw else "kategorik",
        "dtype": str(s.dtype), "Benzersiz": int(s.nunique()),
        "NaN": int(s.isna().sum()),
        "unknown": int((s == "unknown").sum()) if c in cat_cols_raw else 0,
        "unknown %": round(100 * (s == "unknown").mean(), 2) if c in cat_cols_raw else 0.0,
        "Özel kod (pdays=999)": int((s == PDAYS_NO_CONTACT).sum()) if c == "pdays" else 0,
    })
overview_df = pd.DataFrame(overview)
overview_df.to_csv("results/eda_feature_overview.csv", index=False)
overview_df

### Sınıf dağılımı (Şekil 2a)

In [ ]:
cls = y_eda.value_counts().sort_index()
cls_tbl = pd.DataFrame({"Sınıf": ["Negatif (no=0)", "Pozitif (yes=1)"], "n": cls.values,
                        "%": (100 * cls / cls.sum()).round(2).values})
cls_tbl.to_csv("results/eda_class_distribution.csv", index=False)
print(cls_tbl.to_string(index=False))
print(f"\nDengesizlik oranı (neg/pos): {cls[0]/cls[1]:.2f}  |  "
      f"Rastgele sınıflandırıcının beklenen AP'si ≈ pozitif oran = {y_eda.mean():.4f}")

fig, ax = plt.subplots(figsize=(5.5, 4))
bars = ax.bar(cls_tbl["Sınıf"], cls_tbl["n"], color=["#4C72B0", "#DD8452"], edgecolor="black", width=0.55)
for b, n_, p_ in zip(bars, cls_tbl["n"], cls_tbl["%"]):
    ax.text(b.get_x() + b.get_width() / 2, n_ + 30, f"{n_}\n(%{p_})", ha="center", fontweight="bold")
ax.set_ylim(0, cls.max() * 1.2)
ax.set_ylabel("Gözlem sayısı")
ax.set_title(f"Development sınıf dağılımı (n = {len(y_eda)})")
plt.tight_layout(); plt.savefig("figures/fig2a_class_distribution.png", dpi=300); plt.show()

### Sayısal özellikler: tanımlayıcı istatistikler ve aykırı değer işaretleme (Tablo 1b)

In [ ]:
rows = []
for c in num_cols_raw:
    s = X_eda[c].astype(float)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    p1, p99 = s.quantile([0.01, 0.99])
    rows.append({
        "Özellik": c, "Ortalama": s.mean(), "Medyan": s.median(), "SD": s.std(),
        "Q1": q1, "Q3": q3, "IQR": iqr, "Min": s.min(), "Maks": s.max(), "Çarpıklık": s.skew(),
        "IQR aykırı (n)": int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),
        "P1": p1, "P99": p99,
        "P1/P99 dışı (n)": int(((s < p1) | (s > p99)).sum()),
    })
num_summary = pd.DataFrame(rows)
num_summary["IQR aykırı %"] = (100 * num_summary["IQR aykırı (n)"] / len(X_eda)).round(2)
num_summary.round(3).to_csv("results/eda_numeric_summary.csv", index=False)
num_summary.round(3)

**Not (IQR = 0 sorunu):** `pdays` (gözlemlerin ~%96'sı 999) ve `previous` (çoğunluk 0) için IQR = 0'dır. IQR kuralı bu sütunlarda azınlıktaki her değeri "aykırı" sayar. Bu değerler hata değil, anlamlı bilgidir (önceki kampanya teması). Ayrıca RobustScaler, IQR = 0 olduğunda ölçeği 1 alır: ham `pdays` [-999, 0] aralığında kalır. Bu nedenle `pdays = 999` kodu **kural tabanlı** olarak `prev_contacted` (0/1) göstergesine ayrılır ve `pdays` temas yoksa 0 yapılır (`prepare_features`, fit gerektirmez, sızıntı yaratmaz). Kırpma (clipping) bu iki sütuna uygulanmaz. Diğer sütunlarda aykırı değerler **silinmez**; yalnızca nested CV içinde "kırpma yok / eğitim katında öğrenilmiş P1–P99 kırpma" seçeneği olarak denenir.

### Sayısal dağılımlar – histogram (Şekil 2b) ve boxplot (Şekil 2c)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, c in zip(axes.flatten(), num_cols_raw):
    s = X_eda[c]
    if c == "pdays":
        s = s[s != PDAYS_NO_CONTACT]
        ax.set_title(f"pdays (yalnızca ≠ 999, n={len(s)})")
    else:
        ax.set_title(c)
    sns.histplot(s, kde=True, bins=25, ax=ax, color="#4C72B0")
    ax.set_xlabel("")
plt.suptitle("Development – sayısal özellik dağılımları", y=1.0, fontweight="bold")
plt.tight_layout(); plt.savefig("figures/fig2b_numeric_histograms.png", dpi=300); plt.show()

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, c in zip(axes.flatten(), num_cols_raw):
    s = X_eda[c] if c != "pdays" else X_eda.loc[X_eda[c] != PDAYS_NO_CONTACT, c]
    sns.boxplot(x=s, ax=ax, color="#8DA0CB", width=0.5, flierprops={"markersize": 3})
    ax.set_title(c if c != "pdays" else "pdays (≠ 999)")
    ax.set_xlabel("")
plt.suptitle("Development – sayısal özellik boxplot'ları (IQR aykırı noktaları)", y=1.0, fontweight="bold")
plt.tight_layout(); plt.savefig("figures/fig2c_numeric_boxplots.png", dpi=300); plt.show()

### Kategorik özellikler: frekans tabloları ve çubuk grafikler (Şekil 2d)

In [ ]:
freq_rows = []
for c in cat_cols_raw:
    vc = X_eda[c].value_counts()
    for k, v in vc.items():
        freq_rows.append({"Özellik": c, "Kategori": k, "n": int(v), "%": round(100 * v / len(X_eda), 2)})
cat_freq = pd.DataFrame(freq_rows)
cat_freq.to_csv("results/eda_categorical_frequencies.csv", index=False)

fig, axes = plt.subplots(4, 3, figsize=(16, 17))
for ax, c in zip(axes.flatten(), cat_cols_raw):
    vc = X_eda[c].value_counts()
    colors = ["#C44E52" if k == "unknown" else "#4C72B0" for k in vc.index]
    ax.barh(vc.index[::-1], vc.values[::-1], color=colors[::-1], edgecolor="black")
    ax.set_title(f"{c}  (unknown: {(X_eda[c] == 'unknown').sum()})")
    ax.tick_params(axis="y", labelsize=8)
for ax in axes.flatten()[len(cat_cols_raw):]:
    ax.axis("off")
plt.suptitle("Development – kategorik frekanslar (kırmızı = 'unknown')", y=1.0, fontweight="bold")
plt.tight_layout(); plt.savefig("figures/fig2d_categorical_frequencies.png", dpi=300); plt.show()

print(overview_df.loc[overview_df["unknown"] > 0, ["Özellik", "unknown", "unknown %"]].to_string(index=False))

### Korelasyon matrisi (Şekil 2e)

In [ ]:
corr = X_eda[num_cols_raw].corr(method="pearson")
corr.round(3).to_csv("results/eda_correlation_matrix.csv")
plt.figure(figsize=(9, 7.5))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Development – sayısal özellikler Pearson korelasyonu")
plt.tight_layout(); plt.savefig("figures/fig2e_correlation_matrix.png", dpi=300); plt.show()

high = [(a, b, corr.loc[a, b]) for i, a in enumerate(num_cols_raw) for b in num_cols_raw[i + 1:]
        if abs(corr.loc[a, b]) >= 0.8]
print("|r| ≥ 0.80 olan çiftler:")
for a, b, r in sorted(high, key=lambda t: -abs(t[2])):
    print(f"  {a:15s} – {b:15s}: r = {r:+.3f}")

### Seçili özelliklerin hedefe göre betimsel karşılaştırması (Şekil 2f)

In [ ]:
by_y = X_eda[num_cols_raw].assign(y=y_eda).groupby("y").agg(["mean", "median"]).T.round(3)
by_y.columns = ["y=0", "y=1"]
by_y.to_csv("results/eda_numeric_by_target.csv")
display(by_y)

sel = ["euribor3m", "nr.employed", "emp.var.rate", "campaign", "age", "cons.conf.idx"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, c in zip(axes.flatten(), sel):
    sns.boxplot(x=y_eda.map({0: "no (0)", 1: "yes (1)"}), y=X_eda[c], ax=ax,
                hue=y_eda.map({0: "no (0)", 1: "yes (1)"}), palette=["#4C72B0", "#DD8452"],
                legend=False, width=0.5, flierprops={"markersize": 3})
    ax.set_title(c); ax.set_xlabel("y")
plt.suptitle("Hedef sınıfa göre seçili sayısal özellikler (betimsel)", y=1.0, fontweight="bold")
plt.tight_layout(); plt.savefig("figures/fig2f_numeric_by_target.png", dpi=300); plt.show()

# Kategorik: kategori bazında pozitif oranı (betimsel)
rate_rows = []
for c in ["poutcome", "contact", "month", "default"]:
    t = pd.DataFrame({"x": X_eda[c], "y": y_eda}).groupby("x")["y"].agg(["size", "mean"])
    for k, r in t.iterrows():
        rate_rows.append({"Özellik": c, "Kategori": k, "n": int(r["size"]), "Pozitif oranı %": round(100 * r["mean"], 2)})
rate_tbl = pd.DataFrame(rate_rows)
rate_tbl.to_csv("results/eda_categorical_positive_rate.csv", index=False)
rate_tbl

### `pdays = 999` özel kodu (Şekil 2g)

In [ ]:
contacted = X_eda["pdays"] != PDAYS_NO_CONTACT
pd_tbl = pd.DataFrame({
    "Grup": ["pdays = 999 (önceki temas yok)", "pdays ≠ 999 (önceki temas var)"],
    "n": [int((~contacted).sum()), int(contacted.sum())],
    "%": [round(100 * (~contacted).mean(), 2), round(100 * contacted.mean(), 2)],
    "Pozitif oranı %": [round(100 * y_eda[~contacted].mean(), 2), round(100 * y_eda[contacted].mean(), 2)],
})
pd_tbl.to_csv("results/eda_pdays_special_code.csv", index=False)
print(pd_tbl.to_string(index=False))

# Kural tabanlı dönüşümün kontrolü
Xp = prepare_features(dev_df)
print("\nprepare_features sonrası sütunlar:", Xp.shape[1], "| yeni:", [c for c in Xp.columns if c not in X_eda.columns])
print(Xp[["pdays", "prev_contacted"]].describe().round(3))

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.bar(["999 (temas yok)", "≠ 999 (temas var)"], pd_tbl["Pozitif oranı %"], color=["#4C72B0", "#DD8452"], edgecolor="black")
for i, (v, n_) in enumerate(zip(pd_tbl["Pozitif oranı %"], pd_tbl["n"])):
    ax.text(i, v + 1, f"%{v}\n(n={n_})", ha="center")
ax.set_ylabel("Pozitif sınıf oranı (%)"); ax.set_ylim(0, pd_tbl["Pozitif oranı %"].max() * 1.3)
ax.set_title("pdays özel koduna göre pozitif oranı (betimsel)")
plt.tight_layout(); plt.savefig("figures/fig2g_pdays_special_code.png", dpi=300); plt.show()

## 5. EDA bulgularından türetilen test edilebilir deney hipotezleri
Aşağıdaki hipotezler **dış CV sonuçları görülmeden önce** yazılmıştır. Her biri M1–M6 / R1–R6 tasarımındaki belirli bir karşılaştırmayla test edilir. Rapor yazılırken yukarıdaki tablolardaki sayılarla güncelleyin.

| # | Hipotez (EDA gözlemi → beklenti) | Test eden karşılaştırma |
|---|---|---|
| H1 | Pozitif sınıf ~%11'dir (dengesiz). Accuracy, çoğunluk sınıfı modelinde bile ~%89 olur. **Beklenti:** 0.50 eşiğinde MLP'nin Sensitivity değeri düşük, Specificity değeri yüksek kalır. Model sıralaması için AP, Accuracy'den daha ayırt edicidir. | Dummy vs LR vs MLP (dış CV + test) |
| H2 | Sayısal özellik ölçekleri çok farklıdır (`nr.employed` ~5000, `previous` 0–6) ve bazı sütunlarda IQR = 0 / ağır kuyruk vardır. **Beklenti:** StandardScaler, MinMaxScaler'dan daha yüksek iç CV AP verir (MinMax, `campaign` gibi ağır kuyruklu sütunlarda değerleri sıkıştırır). | M1 vs M4 |
| H3 | Makroekonomik değişkenler (`emp.var.rate`, `euribor3m`, `nr.employed`) arasında |r| > 0.9 vardır, yani etkin bilgi boyutu düşüktür. **Beklenti:** İkinci gizli katman ((64,32)) tek katmana ((32,)) göre anlamlı AP artışı sağlamaz, sadece parametre sayısını artırır. | M1 vs M5 |
| H4 | Aynı LR = 1e-3 ile SGD+Momentum, Adam/AdamW'den daha yavaş yakınsar. **Beklenti:** M3'ün en iyi epoch'u ve süresi M1/M2'den belirgin biçimde büyüktür. Daha geniş LR aralığı (1e-3–1e-1) taranan R1–R2'de bu fark azalır. | M1/M2 vs M3; R1–R2 |
| H5 | `campaign` ve `previous` ağır sağ kuyrukludur. **Beklenti:** Eğitim katında öğrenilmiş P1–P99 kırpma, AP'yi en fazla küçük ölçüde değiştirir. | Random Search'teki clip = E/H adayları (betimsel) |
| H6 | Veri küçüktür (dev n ≈ 3.3 bin, iç early-stop kümesinde ~40 pozitif). **Beklenti:** Adam/AdamW modelleri birkaç epoch'ta en iyi doğrulama kaybına ulaşır. Early stopping LR düşüşünden önce devreye girerse scheduler açık/kapalı farkı (M1 vs M6) küçük veya sıfır olur. | M1 vs M6; en iyi epoch kayıtları |

`unknown` stratejisi (ayrı kategori / en sık sınıf) ve clipping, EDA'ya bakılarak **seçilmemiş**, nested CV içinde aday boyutu olarak bırakılmıştır.

## 6. Bu defterin ürettiği dosyalar

In [ ]:
for f in ["configs/seeds.json", "results/system_info.json", "results/duplicate_check.csv",
          "results/dataset_split.csv", "results/dataset_split_summary.csv",
          "results/eda_feature_overview.csv", "results/eda_numeric_summary.csv",
          "results/eda_categorical_frequencies.csv", "results/eda_correlation_matrix.csv",
          "results/eda_numeric_by_target.csv", "results/eda_categorical_positive_rate.csv",
          "results/eda_pdays_special_code.csv", "data/dev_data.csv", "data/test_data.csv"]:
    print(("✓ " if os.path.exists(f) else "✗ ") + f)
print("Şekiller:", sorted(f for f in os.listdir("figures") if f.startswith("fig2")))